# Operations Program KPI Dashboard
**Keandra Morris | Operations Project Manager**  
Portfolio: [keandrajm.github.io/PM_Portfolio](https://keandrajm.github.io/PM_Portfolio/)  

---
This notebook builds a multi-panel KPI dashboard modeled after the Excel reporting tools
used to track a $500K annual operations program at Airstreams Renewables, Inc.

**Dashboard panels:**
- Four executive KPI summary cards (Budget Utilization, Compliance, Satisfaction, Waste Reduction)
- Budget vs. Actual spend by month
- Compliance score and user satisfaction dual-axis trend
- Manual task volume reduction (automation impact)
- Waste percentage reduction trend

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

GREEN = '#1B4332'
GOLD  = '#B8933A'
MUTED = '#4A5568'
LGRAY = '#9CA3AF'
TEAL  = '#2980B9'
RED   = '#C0392B'
WHITE = '#FFFFFF'
BG    = '#E8E8E8'

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df = pd.DataFrame({
    'Month':          months,
    'Budget_Spent':   [38500,39200,41000,40500,42000,41800,39900,41200,43000,44100,45000,44500],
    'Budget_Target':  [41000]*12,
    'Compliance':     [72,78,82,85,87,90,88,91,93,94,96,97],
    'Satisfaction':   [3.6,3.7,3.8,3.9,4.0,4.1,4.1,4.2,4.3,4.4,4.5,4.6],
    'Manual_Tasks':   [142,135,118,102,89,76,68,62,58,54,52,50],
    'Waste_Pct':      [8.2,7.8,7.1,6.8,6.2,5.9,5.7,5.4,5.1,4.8,4.5,4.3],
})

df['Budget_Variance']  = df['Budget_Target'] - df['Budget_Spent']
df['Task_Reduction_Pct'] = ((142 - df['Manual_Tasks']) / 142 * 100).round(1)

# KPI summary
budget_util = df['Budget_Spent'].sum() / (df['Budget_Target'].sum()) * 100
compliance_final = df['Compliance'].iloc[-1]
satisfaction_final = df['Satisfaction'].iloc[-1]
waste_reduction = (df['Waste_Pct'].iloc[0] - df['Waste_Pct'].iloc[-1]) / df['Waste_Pct'].iloc[0] * 100

print('KPI SUMMARY')
print('─' * 35)
print(f'  Budget Utilization:     {budget_util:.1f}%')
print(f'  Final Compliance Score: {compliance_final} pts (+{compliance_final-72} pts gain)')
print(f'  User Satisfaction:      {satisfaction_final}/5.0 (+{satisfaction_final-3.6:.1f} pts gain)')
print(f'  Waste Reduction:        {waste_reduction:.1f}%')
print(f'  Total Budget Savings:   ${df["Budget_Variance"].sum():,.0f}')
print(f'  Task Reduction:         {df["Task_Reduction_Pct"].iloc[-1]}%')

## 2. Full KPI Dashboard

In [ ]:
fig = plt.figure(figsize=(14, 9), facecolor=BG)
fig.patch.set_facecolor(BG)

x = range(12)

# Title bar
title_ax = fig.add_axes([0, 0.93, 1, 0.07])
title_ax.set_facecolor(GREEN)
title_ax.text(0.5, 0.55, 'Operations Program KPI Dashboard  ·  FY 2024  ·  Airstreams Renewables',
              ha='center', va='center', color='white', fontsize=13, fontweight='bold')
title_ax.set_xticks([]); title_ax.set_yticks([])
for sp in title_ax.spines.values(): sp.set_visible(False)

# KPI Cards
kpi_data = [
    ('Budget Utilization', f'{budget_util:.1f}%', '$500K managed', GOLD),
    ('Compliance Score',   f'{compliance_final} pts', f'+{compliance_final-72} pts improvement', GREEN),
    ('User Satisfaction',  f'{satisfaction_final} / 5', f'+{satisfaction_final-3.6:.1f} pts improvement', TEAL),
    ('Waste Reduction',    f'{waste_reduction:.1f}%', f'From 8.2% to 4.3%', '#7B3F00'),
]
for i, (title, value, sub, color) in enumerate(kpi_data):
    ax_c = fig.add_axes([0.01 + i*0.248, 0.78, 0.235, 0.14])
    ax_c.set_facecolor(WHITE)
    ax_c.set_xticks([]); ax_c.set_yticks([])
    for sp in ax_c.spines.values(): sp.set_color('#D1D5DB')
    ax_c.axvline(x=0, ymin=0, ymax=1, linewidth=5, color=color)
    ax_c.text(0.08, 0.78, title, transform=ax_c.transAxes, fontsize=8.5, color=MUTED)
    ax_c.text(0.08, 0.35, value, transform=ax_c.transAxes, fontsize=20, color=color, fontweight='bold')
    ax_c.text(0.08, 0.08, sub, transform=ax_c.transAxes, fontsize=7.5, color=LGRAY)

# Budget vs Actual
ax1 = fig.add_axes([0.03, 0.42, 0.56, 0.34], facecolor=WHITE)
xn = np.arange(12)
ax1.bar(xn-0.2, [b/1000 for b in df['Budget_Target']], 0.38, label='Budget', color='#CBD5E0', edgecolor='none')
ax1.bar(xn+0.2, [b/1000 for b in df['Budget_Spent']],  0.38, label='Actual', color=GREEN, alpha=0.85, edgecolor='none')
ax1.set_xticks(xn); ax1.set_xticklabels(months, fontsize=7.5)
ax1.set_ylabel('$K', fontsize=8); ax1.set_ylim(35, 47)
ax1.set_title('Budget vs. Actual Spend ($K)', fontsize=9, fontweight='bold', color='#1C1C1C', loc='left', pad=6)
ax1.legend(fontsize=7.5, frameon=False)
ax1.grid(axis='y', alpha=0.25, linewidth=0.7)
ax1.tick_params(labelsize=7.5)
for sp in ['top','right']: ax1.spines[sp].set_visible(False)

# Compliance + Satisfaction
ax2 = fig.add_axes([0.63, 0.42, 0.35, 0.34], facecolor=WHITE)
ax2b = ax2.twinx()
ax2.plot(x, df['Compliance'], color=GOLD, linewidth=2.2, marker='o', markersize=4,
         markerfacecolor=WHITE, markeredgewidth=1.5, markeredgecolor=GOLD, label='Compliance (L)')
ax2b.plot(x, df['Satisfaction'], color=TEAL, linewidth=2.2, marker='s', markersize=4,
          markerfacecolor=WHITE, markeredgewidth=1.5, markeredgecolor=TEAL, label='Satisfaction (R)')
ax2.set_xticks(list(x)); ax2.set_xticklabels(months, fontsize=7.5)
ax2.set_ylabel('Compliance Score', fontsize=7.5, color=GOLD)
ax2b.set_ylabel('Satisfaction /5', fontsize=7.5, color=TEAL)
ax2.set_ylim(60, 105); ax2b.set_ylim(2.5, 5.5)
ax2.set_title('Compliance & Satisfaction Trends', fontsize=9, fontweight='bold', color='#1C1C1C', loc='left', pad=6)
ax2.tick_params(labelsize=7.5); ax2b.tick_params(labelsize=7.5)
ax2.grid(axis='y', alpha=0.25, linewidth=0.7)
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, fontsize=7, frameon=False, loc='upper left')
for sp in ['top']: ax2.spines[sp].set_visible(False)

# Manual task reduction
ax3 = fig.add_axes([0.03, 0.05, 0.37, 0.33], facecolor=WHITE)
ax3.fill_between(x, df['Manual_Tasks'], alpha=0.15, color=RED)
ax3.plot(x, df['Manual_Tasks'], color=RED, linewidth=2.2, marker='o', markersize=4,
         markerfacecolor=WHITE, markeredgewidth=1.5, markeredgecolor=RED)
ax3.set_xticks(list(x)); ax3.set_xticklabels(months, fontsize=7.5)
ax3.set_ylabel('Tasks / Month', fontsize=8)
ax3.set_title('Manual Task Volume (Automation Impact)', fontsize=9, fontweight='bold', color='#1C1C1C', loc='left', pad=6)
ax3.grid(axis='y', alpha=0.25, linewidth=0.7)
ax3.tick_params(labelsize=7.5)
for sp in ['top','right']: ax3.spines[sp].set_visible(False)
ax3.text(11, 52, '65% reduced', fontsize=7.5, color=RED, fontweight='bold', ha='right')

# Waste % trend
ax4 = fig.add_axes([0.44, 0.05, 0.37, 0.33], facecolor=WHITE)
ax4.fill_between(x, df['Waste_Pct'], alpha=0.15, color=GREEN)
ax4.plot(x, df['Waste_Pct'], color=GREEN, linewidth=2.2, marker='o', markersize=4,
         markerfacecolor=GOLD, markeredgewidth=1.5, markeredgecolor=GREEN)
ax4.set_xticks(list(x)); ax4.set_xticklabels(months, fontsize=7.5)
ax4.set_ylabel('Waste %', fontsize=8)
ax4.set_title('Waste Reduction Trend (%)', fontsize=9, fontweight='bold', color='#1C1C1C', loc='left', pad=6)
ax4.grid(axis='y', alpha=0.25, linewidth=0.7)
ax4.tick_params(labelsize=7.5)
for sp in ['top','right']: ax4.spines[sp].set_visible(False)
ax4.annotate(f'{waste_reduction:.1f}% total\nreduction', xy=(11, 4.3), xytext=(8.5, 7.5),
             fontsize=7.5, color=GREEN, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2))

# Summary panel
ax5 = fig.add_axes([0.83, 0.05, 0.155, 0.33], facecolor=WHITE)
ax5.set_xticks([]); ax5.set_yticks([])
for sp in ax5.spines.values(): sp.set_color('#D1D5DB')
summary = [
    ('Total Budget Savings', f'${df["Budget_Variance"].sum():,.0f}'),
    ('Compliance Gain',       f'+{compliance_final-72} pts'),
    ('Task Reduction',        f'{df["Task_Reduction_Pct"].iloc[-1]}%'),
    ('Waste Saved',          '$5,200+'),
    ('Satisfaction Gain',     f'+{satisfaction_final-3.6:.1f} pts'),
    ('Program Duration',      '6 yrs'),
]
ax5.text(0.5, 0.96, 'Program Summary', ha='center', va='top',
         fontsize=8, fontweight='bold', color=GREEN, transform=ax5.transAxes)
for i, (k, v) in enumerate(summary):
    y_pos = 0.83 - i*0.14
    ax5.text(0.06, y_pos, k, ha='left', va='center', fontsize=7, color=MUTED, transform=ax5.transAxes)
    ax5.text(0.94, y_pos, v, ha='right', va='center', fontsize=7.5, color='#1C1C1C',
             fontweight='bold', transform=ax5.transAxes)
    if i < 5:
        ax5.axhline(y=(0.79 - i*0.14), xmin=0.04, xmax=0.96, linewidth=0.5, color='#E5E7EB')

fig.text(0.5, 0.005, 'Keandra Morris  ·  Operations Project Manager  ·  keandrajm.github.io/PM_Portfolio/',
         ha='center', fontsize=7.5, color=LGRAY, style='italic')

plt.show()

---
**Notebook author:** Keandra Morris  
**Portfolio:** [keandrajm.github.io/PM_Portfolio](https://keandrajm.github.io/PM_Portfolio/)  
**Tools used:** Python · Pandas · NumPy · Matplotlib